In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

df = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_st_test_corpus_data.parquet")

In [3]:
df.head()

,query,context,type,synthesized,source,metadata,url1
727493,SHEBA Ice Camp environmental monitoring,Description: NCAR portable automated mesonet (...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214601988-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
489836,bio-geochemical cycles Ross Sea,Description: The data sets include measurement...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214593766-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
869447,Pearl Harbor oceanographic data,Description: This dataset contains oceanograph...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2089378855-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
668676,optical backscatter measurement techniques,Description: Two hydrographic surveys were per...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214155000-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
723711,gravity and magnetic field data integration,Description: This data set contains underway g...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214611760-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...


In [4]:
df.shape

(128455, 7)

In [5]:
# sperating out the meta data

# for query
# - type
# - synthesized


# for cocrpus
# - source
# - url0

df.drop(columns=["metadata"], inplace=True)

In [6]:
df.columns

Index(['query', 'context', 'type', 'synthesized', 'source', 'url1'], dtype='object')

# converting it to standard jsonal format

In [7]:
import pandas as pd
import json
import os

def create_jsonl_and_qrels(df: pd.DataFrame, output_dir: str):
    """
    Converts a DataFrame into JSONL format for queries and corpus,
    and a TSV file for query-relevance pairs.

    The function creates the following structure:
    output_dir/
    ├── corpus.jsonl
    ├── queries.jsonl
    └── qrels/
        └── test.tsv

    Args:
        df (pd.DataFrame): The input DataFrame with columns:
                           'query', 'context', 'type', 'synthesized',
                           'source', 'url1'.
        output_dir (str): The directory where the output files will be saved.
    """
    # --- 1. Create Output Directories ---
    qrels_dir = os.path.join(output_dir, 'qrels')
    if not os.path.exists(qrels_dir):
        os.makedirs(qrels_dir)
        print(f"Created directory: {qrels_dir}")

    # --- 2. Process Corpus ---
    # Get unique contexts to create the corpus
    corpus_df = df[['context', 'source', 'url1']].drop_duplicates(subset=['context']).reset_index(drop=True)
    
    # Create a mapping from context text to a unique corpus ID
    context_to_id = {row['context']: f"c{index}" for index, row in corpus_df.iterrows()}
    
    corpus_filepath = os.path.join(output_dir, 'corpus.jsonl')
    print(f"Generating {corpus_filepath}...")
    with open(corpus_filepath, 'w') as f:
        for index, row in corpus_df.iterrows():
            corpus_id = context_to_id[row['context']]
            corpus_entry = {
                "_id": corpus_id,
                "text": row['context'],
                "metadata": {
                    "source": row['source'],
                    "url": row['url1']
                }
            }
            f.write(json.dumps(corpus_entry) + '\n')
    print(f"Successfully created {corpus_filepath} with {len(corpus_df)} entries.")

    # --- 3. Process Queries ---
    # Get unique queries. Per user, queries will already be unique.
    queries_df = df[['query', 'type', 'synthesized']].drop_duplicates(subset=['query']).reset_index(drop=True)

    # Create a mapping from query text to a unique query ID
    query_to_id = {row['query']: f"q{index}" for index, row in queries_df.iterrows()}

    queries_filepath = os.path.join(output_dir, 'queries.jsonl')
    print(f"\nGenerating {queries_filepath}...")
    with open(queries_filepath, 'w') as f:
        for index, row in queries_df.iterrows():
            query_id = query_to_id[row['query']]
            query_entry = {
                "_id": query_id,
                "text": row['query'],
                "metadata": {
                    "type": row['type'],
                    "synthesized": row['synthesized']
                }
            }
            f.write(json.dumps(query_entry) + '\n')
    print(f"Successfully created {queries_filepath} with {len(queries_df)} entries.")


    # --- 4. Create Qrels (Query-Relevance) File ---
    qrels_filepath = os.path.join(qrels_dir, 'test.tsv')
    print(f"\nGenerating {qrels_filepath}...")
    
    # Create a list to hold the relationship data
    qrels_data = []
    # Use the original dataframe to preserve all query-context relationships
    for index, row in df.iterrows():
        query_id = query_to_id.get(row['query'])
        corpus_id = context_to_id.get(row['context'])
        
        if query_id and corpus_id:
            # The format is: query-id, corpus-id, score (assuming 1)
            qrels_data.append([query_id, corpus_id, 1])

    # Create a DataFrame for qrels and save to TSV without duplicates
    qrels_df = pd.DataFrame(qrels_data, columns=['query-id', 'corpus-id', 'score'])
    qrels_df.drop_duplicates(inplace=True)
    qrels_df.to_csv(qrels_filepath, sep='\t', index=False, header=True)
    
    print(f"Successfully created {qrels_filepath} with {len(qrels_df)} relations.")


In [8]:
output_dir = "/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/"

create_jsonl_and_qrels(df, output_dir=output_dir)

Created directory: /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/qrels
Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/corpus.jsonl...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/corpus.jsonl with 67442 entries.

Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/queries.jsonl...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/queries.jsonl with 113532 entries.

Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/qrels/test.tsv...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_IR_benchmark/qrels/test.tsv with 125895 relations.


# testiong the jsonl and qrels files